Celda 1: Importación de librerías y configuración de rutas

In [3]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

PATH_INPUT = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\Dataset_ML.xlsx"
PATH_OUTPUT_DIR = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset"

os.makedirs(PATH_OUTPUT_DIR, exist_ok=True)

print("Librerías cargadas y rutas configuradas.")

Librerías cargadas y rutas configuradas.


Celda 2: Carga de datos y limpieza inicial de columnas

In [4]:
df_raw = pd.read_excel(PATH_INPUT)

COLS_EXCLUIR = ['Nro', 'Contratante', 'Dirección', 'DNI', 'Teléfono', 'Fallecido', 'Velatorio']
df = df_raw.drop(columns=COLS_EXCLUIR)

COLS_BINARIAS = ['Carroza', 'Carroza flores', 'Auto', 'Microbus']

print(f"Dataset cargado: {df.shape[0]} registros y {df.shape[1]} columnas.")

Dataset cargado: 14 registros y 11 columnas.


Celda 3: Tratamiento de Fechas (Incluyen 2022-2025)

In [5]:
df['Fecha'] = df['Fecha'].replace("Vacío", np.nan)

df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')

df['Fecha'] = df['Fecha'].ffill().bfill()

df = df.sort_values(by='Fecha').reset_index(drop=True)

print("Fechas limpias e imputadas. Rango detectado:", df['Fecha'].min(), "a", df['Fecha'].max())

Fechas limpias e imputadas. Rango detectado: 2022-08-12 00:00:00 a 2025-08-22 00:00:00


Celda 4: Corrección de Monto y Outliers

In [6]:
df['Monto'] = df['Monto'].replace(207000, 2070.00)

Q1 = df['Monto'].quantile(0.25)
Q3 = df['Monto'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 3 * IQR

df['monto_outlier'] = df['Monto'] > limite_superior
print(f"Monto corregido. Registros que aún exceden el límite lógico (>{limite_superior}): {df['monto_outlier'].sum()}")

Monto corregido. Registros que aún exceden el límite lógico (>11247.5): 0


Celda 5: Normalización de Categorías

In [7]:
def limpiar_texto(val):
    if pd.isna(val) or str(val).lower() == 'vacío':
        return "no_especificado"
    return str(val).strip().lower()

ATAUD_MAP = {
    'lincol': 'lincoln',
    'imperim': 'imperial',
    'impor.': 'imperial',
    'ardiente iluminada': 'iluminada',
    'vacio': 'no_especificado'
}

for col in ['Ataud_Modelo', 'Ataud_Color', 'Capilla', 'Forma de pago']:
    df[col] = df[col].apply(limpiar_texto)
    df[col] = df[col].replace(ATAUD_MAP)
    df[col] = df[col].str.title()

for col in COLS_BINARIAS:
    df[col] = df[col].astype(str).str.lower().str.strip().map({'si': 1, 'no': 0}).fillna(0).astype(int)

df['Ataud_completo'] = df['Ataud_Modelo'] + "_" + df['Ataud_Color']

print("Categorías normalizadas y feature 'Ataud_completo' creado.")

Categorías normalizadas y feature 'Ataud_completo' creado.


Celda 6: Agregación Mensual

In [8]:
df['Periodo'] = df['Fecha'].dt.to_period('M')

df_mensual = df.groupby('Periodo').agg(
    servicios_totales=('Fecha', 'count'),
    monto_total=('Monto', 'sum'),
    monto_promedio=('Monto', 'mean'),
    carroza_count=('Carroza', 'sum'),
    carroza_flores_count=('Carroza flores', 'sum'),
    auto_count=('Auto', 'sum'),
    microbus_count=('Microbus', 'sum'),
    cargadores_total=('Cargadores', 'sum')
).reset_index()

all_months = pd.period_range(start=df['Periodo'].min(), end=df['Periodo'].max(), freq='M')
df_mensual = df_mensual.set_index('Periodo').reindex(all_months, fill_value=0).reset_index()
df_mensual.rename(columns={'index': 'Periodo'}, inplace=True)

print(f"Dataset mensual creado con {len(df_mensual)} meses (incluyendo huecos con valor 0).")

Dataset mensual creado con 37 meses (incluyendo huecos con valor 0).


Celda 7: Exportación de resultados

In [9]:
df.to_excel(os.path.join(PATH_OUTPUT_DIR, "dataset_limpio.xlsx"), index=False)
df_mensual.to_excel(os.path.join(PATH_OUTPUT_DIR, "dataset_mensual.xlsx"), index=False)

metricas = pd.DataFrame({
    'Métrica': ['Total Registros', 'Rango Meses', 'Suma Montos', 'Promedio Servicios/Mes'],
    'Valor': [len(df), len(df_mensual), df['Monto'].sum(), df_mensual['servicios_totales'].mean()]
})
metricas.to_excel(os.path.join(PATH_OUTPUT_DIR, "metricas_preprocesamiento.xlsx"), index=False)

print("Archivos exportados exitosamente en:", PATH_OUTPUT_DIR)

Archivos exportados exitosamente en: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset
